# OASIS-2 MRI Feature Extraction
**Purpose:** Extract 512-dimensional feature embeddings from every brain scan using a 3D ResNet-18 (MONAI), and save them to Google Drive.

- Input : `MyDrive/DeepBET_Output/` — original MRI files, **never modified**
- Output: `MyDrive/OASIS_Features/{patient_id}/{mri_id}.npy` — one `.npy` per visit

**Run cells top to bottom. GPU runtime required.**

In [ ]:
!pip install monai nibabel --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted!


In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from monai.networks.nets import resnet
from tqdm import tqdm

warnings.filterwarnings("ignore")
print("All imports successful!")

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


All imports successful!


In [ ]:
DATA_PATH = Path("/content/drive/MyDrive/preprocessed/DeepBET_Output")

FEATURES_PATH = Path("/content/drive/MyDrive/new_features_1")
TRAIN_FEATURES = FEATURES_PATH / "train"
TEST_FEATURES  = FEATURES_PATH / "test"

TRAIN_FEATURES.mkdir(exist_ok=True)
TEST_FEATURES.mkdir(exist_ok=True)

TARGET_SHAPE = (64, 64, 64)

BATCH_SIZE  = 4
NUM_WORKERS = 2

KNOWN_LABELS = {"demented", "non_demented", "converted"}

FEATURES_PATH.mkdir(parents=True, exist_ok=True)

print(f"Source  : {DATA_PATH}")
print(f"Output  : {FEATURES_PATH}")
print(f"Exists  : {DATA_PATH.exists()}")

Source  : /content/drive/MyDrive/preprocessed/DeepBET_Output
Output  : /content/drive/MyDrive/new_features_1
Exists  : True


In [ ]:
label_map = {
    "non_demented": 0,
    "converted": 1,
    "demented": 2
}


def find_all_scans(root: Path) -> list:

    scans = []

    for patient_folder in sorted(root.iterdir()):

        if not patient_folder.is_dir():
            continue

        patient_id = patient_folder.name

        for label_folder in sorted(patient_folder.iterdir()):

            if not label_folder.is_dir():
                continue

            class_name = label_folder.name.lower()

            if class_name not in label_map:
                continue

            # numeric label
            label = label_map[class_name]

            brain_folder = label_folder / "brain"

            if not brain_folder.exists():
                continue

            for scan_file in sorted(brain_folder.glob("*.nii.gz")):

                fname = scan_file.name


                mri_id = fname.split("_mpr")[0]

                scans.append({
                    "patient_id": patient_id,
                    "mri_id": mri_id,
                    "label": label,
                    "label_name": class_name,
                    "filepath": scan_file,
                })

    return scans


all_scans = find_all_scans(DATA_PATH)

from collections import Counter

label_counts = Counter(s["label_name"] for s in all_scans)

print(f"Total scans found : {len(all_scans)}")

print(f"Unique patients   : {len(set(s['patient_id'] for s in all_scans))}")

print("\nScans per class:")

for label, count in label_counts.items():
    print(f"  {label:>15}: {count}")

Total scans found : 1368
Unique patients   : 150

Scans per class:
     non_demented: 692
         demented: 542
        converted: 134


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# Get unique patients
all_patients = sorted(
    list(set(scan["patient_id"] for scan in all_scans))
)

# Split patients
train_patients, test_patients = train_test_split(
    all_patients,
    test_size=0.2,
    random_state=42
)

# Build scan lists
train_scans = [
    s for s in all_scans
    if s["patient_id"] in train_patients
]

test_scans = [
    s for s in all_scans
    if s["patient_id"] in test_patients
]

print("Train patients:", len(train_patients))
print("Test patients :", len(test_patients))

print("Train scans:", len(train_scans))
print("Test scans :", len(test_scans))

Train patients: 120
Test patients : 30
Train scans: 1093
Test scans : 275


In [ ]:
def preprocess_volume(filepath: Path) -> torch.Tensor:
    """
    Load a .nii.gz brain scan and return a (1, D, H, W) float32 tensor.
    Steps:
        load
        → squeeze extra dims
        → z-score normalise
        → resize to TARGET_SHAPE
    """

    img = nib.load(str(filepath))
    volume = img.get_fdata().astype(np.float32)

    # Squeeze out extra trailing dimensions
    while volume.ndim > 3:
        volume = volume.squeeze(-1)

    # Z-score normalization
    mean, std = volume.mean(), volume.std()

    if std > 0:
        volume = (volume - mean) / std

    # (D,H,W) → (1,1,D,H,W)
    volume = torch.tensor(volume).unsqueeze(0).unsqueeze(0)

    # Resize volume
    volume = F.interpolate(
        volume,
        size=TARGET_SHAPE,
        mode="trilinear",
        align_corners=False
    )

    # (1,1,D,H,W) → (1,D,H,W)
    return volume.squeeze(0)


class OASISDataset(Dataset):
    """
    Wraps the scan list.

    Returns:
        volume_tensor,
        label,
        scan_index
    """

    def __init__(self, scan_list: list):
        self.scans = scan_list

    def __len__(self):
        return len(self.scans)

    def __getitem__(self, idx):

        scan = self.scans[idx]

        volume = preprocess_volume(scan["filepath"])

        label = scan["label"]

        return volume, label, idx


train_dataset = OASISDataset(train_scans)

test_dataset = OASISDataset(test_scans)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Train scans : {len(train_dataset)}")
print(f"Test scans  : {len(test_dataset)}")

Train scans : 1093
Test scans  : 275


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cpu":
    print("⚠️  WARNING: No GPU detected. Extraction will be very slow. "
          "Go to Runtime → Change runtime type → GPU.")

def build_feature_extractor() -> nn.Module:
    model = resnet.resnet18(
        spatial_dims     = 3,
        n_input_channels = 1,
        num_classes      = 512,  # placeholder — fc will be replaced below
    )
    # Remove the classification head → output is now the 512-dim avgpool vector
    model.fc = nn.Linear(model.fc.in_features, 3)
    return model

model = build_feature_extractor().to(device)
model.eval()  # disables dropout / batchnorm training behaviour

# Quick sanity check — pass a dummy volume through
with torch.no_grad():
    dummy = torch.zeros(1, 1, *TARGET_SHAPE).to(device)
    out   = model(dummy)
    print(f"Output shape for one scan: {out.shape}")
    #assert out.shape == (1, 512), f"Expected (1, 512), got {out.shape}"
    print("Feature extractor ready! Each scan → 512-dim vector.")

Device: cuda
Output shape for one scan: torch.Size([1, 3])
Feature extractor ready! Each scan → 512-dim vector.


In [ ]:
# Freeze EVERYTHING first
for param in model.parameters():
    param.requires_grad = False


# Unfreeze layer4
for param in model.layer4.parameters():
    param.requires_grad = True


# Unfreeze final classifier
for param in model.fc.parameters():
    param.requires_grad = True

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

epochs = 15

model.to(device)

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for volumes, labels, _ in train_loader:

        volumes = volumes.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(volumes)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)

    print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}")

KeyboardInterrupt: 

In [ ]:
model.fc = nn.Identity()

model.eval()

dummy = torch.randn(1, 1, 128, 128, 128).to(device)

with torch.no_grad():
    out = model(dummy)

print(out.shape)

In [ ]:
from collections import defaultdict

def extract_and_save(model, loader, scan_list, features_path):
    model.eval()
    # Dictionary to hold lists of features: { "mri_id": [feat1, feat2, ...] }
    session_features = defaultdict(list)

    print("Step 1: Extracting features from all scans...")
    with torch.no_grad():
        for batch_volumes, batch_labels, batch_indices in tqdm(loader, desc="Extracting"):
            batch_volumes = batch_volumes.to(device)
            features = model(batch_volumes).cpu().numpy()

            for i, idx in enumerate(batch_indices.tolist()):
                scan = scan_list[idx]
                # Groups by session (e.g., OAS2_0009_MR1)
                mri_id = scan["mri_id"]
                session_features[mri_id].append(features[i])

    print("\nStep 2: Averaging and saving one file per session...")
    saved = 0
    for mri_id, feat_list in tqdm(session_features.items(), desc="Saving"):
        # Find the patient_id from the first scan in this session
        # (This assumes mri_id starts with patient_id)
        patient_id = mri_id.split("_MR")[0]

        patient_out_folder = features_path / patient_id
        patient_out_folder.mkdir(parents=True, exist_ok=True)

        # Calculate the MEAN of all mpr scans for this visit
        avg_feature = np.mean(feat_list, axis=0) # Shape stays (512,)

        save_path = patient_out_folder / f"{mri_id}.npy"
        np.save(save_path, avg_feature)
        saved += 1

    print(f"\n✅ Successfully averaged {len(session_features)} sessions and saved to {features_path}")

# Extract train features
extract_and_save(
    model,
    train_loader,
    train_scans,
    TRAIN_FEATURES
)

# Extract test features
extract_and_save(
    model,
    test_loader,
    test_scans,
    TEST_FEATURES
)




In [ ]:
# Count unique expected sessions per patient
expected_sessions = defaultdict(set)
for s in all_scans:
    expected_sessions[s["patient_id"]].add(s["mri_id"])

saved_counts = defaultdict(int)
all_npy = list(FEATURES_PATH.rglob("*.npy"))
for f in all_npy:
    saved_counts[f.parent.name] += 1

print(f"Total .npy sessions saved : {len(all_npy)}")
print(f"Total unique sessions expected : {sum(len(v) for v in expected_sessions.values())}")

# Report mismatches based on sessions, not raw scans
mismatches = [
    pid for pid in expected_sessions
    if len(expected_sessions[pid]) != saved_counts.get(pid, 0)
]

if not mismatches:
    print("\n✅ All patients have the correct number of averaged session files!")

Total .npy sessions saved : 0
Total unique sessions expected : 374


In [ ]:
!rm -rf /content/drive/MyDrive/OASIS_Features/*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Label mapping
# converted is treated as demented
label_map = {
    "demented": 0,
    "converted": 0,
    "non_demented": 1
}

# Load pretrained ResNet-18
model = models.resnet18(pretrained=True)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last two residual blocks
for param in model.layer3.parameters():
    param.requires_grad = True

for param in model.layer4.parameters():
    param.requires_grad = True

# Add classification head
num_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 2)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

print(model)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 136MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
model.train()

for images, labels in train_loader:

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)

    loss = criterion(outputs, labels)

    loss.backward()

    optimizer.step()

    preds = torch.argmax(outputs, dim=1)

    correct = (preds == labels).sum().item()

    print("Batch Accuracy:", correct / labels.size(0))


ValueError: too many values to unpack (expected 2)